# 36. The three no-encoder views

Three arms, one per learner, each on the **raw 12 columns with no encoder at all**. Each is
one variable against its own target-encoded counterpart in the ledger, and the variable is
the same one in all three cases: **the encoder, on or off.**

| arm | baseline it changes one variable against | that row's CV |
|---|---|---|
| `lgb_raw` | row 17 `lgbm_bag08_seed42_te` | 0.966782 |
| `xgb_raw` | row 38 `xgb_te` | 0.967099 |
| `cat_raw` | row 26 `catboost_te` | 0.966915 |

Same folds (sha `ec282b0968059676`), same seed, same budget, same learner settings as the
row each arm is compared to. Only the feature set changes: 36 encoded features become 12
raw ones with native categorical and native NaN handling.

## These arms are expected to LOSE, and that is not the point

`lgb_raw` is row 9's configuration exactly, so it should reproduce **0.963471**, and that
reproduction is a check this notebook runs rather than a result it reports. All three arms
should land roughly 0.003 below their encoded counterparts, because that is what target
encoding was worth in row 17 against row 9.

**The point is what a fitted combiner does with them, and this repo has the evidence that
those are different questions.** Row 33 put a model 0.0247 behind the pack into the stack
with a **+0.1178** coefficient. This repo records the correction bluntly: a member's own CV
is close to irrelevant to whether a combiner wants it, and the relevant statistic is
disagreement. Every one of the 23 member vectors this repo has saved consumes the same
encoder, so the stack has never once been offered a genuinely different view of the data.

## Where the idea came from, and its CV scheme

srcJ's `s6e8-diversity-beats-strength` fits a nested rank-gauss logistic stack over a
177-model pool and publishes the coefficient table. **Three of the top four members are the
no-encoder views**, arrived at independently by XGBoost, LightGBM and CatBoost:

| their member | their solo OOF | their coefficient |
|---|---|---|
| CatBoost, its own ordered target statistics | 0.968601 | +0.749 |
| XGBoost, **no target encoding** | 0.967042 | +0.334 |
| LightGBM, **no target encoding** | 0.966381 | +0.278 |
| CatBoost, **no target encoding** | 0.968405 | +0.176 |
| XGBoost + target encoding | 0.968372 | +0.158 |

The bottom of their table is seeds and hyperparameter tweaks of recipes already present,
taking small negative coefficients. That is the same shape as this repo's row 28, where five
members were carrying nothing and were pruned.

Their fold split is `StratifiedKFold(5, shuffle=True, random_state=42)` over `train.csv` in
original file row order, which was regenerated here and matched against the recorded sha
rather than taken on trust. It is bit-identical to ours.

**What does not transfer.** Their "no target encoding" arms still carry an imputation and
composition-ratio feature block, so their absolute numbers are not ours to expect. Only the
*ordering* claim is being borrowed, and it is being re-measured on our folds.

## The prediction, written before the run

- All three arms land **0.002 to 0.004 below** their encoded counterparts.
- `lgb_raw` reproduces row 9 to within 1e-6.
- **In `37`, at least one raw view earns a coefficient in the top half of the member table**,
  despite being the three weakest members in it.

The third is the one that can be wrong, and it is the reason to run this. If all three raw
views take coefficients near zero, then this repo's stack really was saturated and rows 55
to 65 were measuring a genuine ceiling rather than a lack of variety.

## What this decides

Nothing on its own. It writes six member vectors. Membership is `37_stack_views.ipynb`, a
separate ledger row, because it changes a different variable. No submission csv here.

In [ ]:
# One flag. The run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42

# The budget convention every arm's baseline used: learning_rate * n_estimators = 100.
LR = 0.05
N_EST = 2000
MAX_DEPTH = 6
BENCH_EST = 200
PROBE_FOLD = 0

ARMS = ["lgb_raw", "xgb_raw", "cat_raw"]

# Measured 2026-08-19 and written up: at 16,000 rows this machine runs 101x
# slower at all-threads than at one thread. The baselines ran on Kaggle at -1, so -1 is
# what matches them. A smoke run produces no ledger number, so overriding it costs nothing.
THREADS = -1

MAX_HOURS = 9.0

# The row each arm changes one variable against: the encoder, on or off.
BASELINES = {
    "lgb_raw": ("te_bag42", 0.966782, "row 17 lgbm_bag08_seed42_te"),
    "xgb_raw": ("xgb_te", 0.967099, "row 38 xgb_te"),
    "cat_raw": ("catboost_te", 0.966915, "row 26 catboost_te"),
}
# lgb_raw is row 9's configuration exactly, so it has a number it must hit.
ROW9_CV = 0.963471
EXPECTED_FOLD_SHA = "ec282b0968059676"

print(f"SMOKE = {SMOKE}   lr {LR}   n_estimators {N_EST}")
print(f"arms: {ARMS}")

## Stage 1. Data, folds, leak checklist

Lifted from `34_xgb_depth.ipynb`. The fold checksum is checked before anything trains,
because a misaligned out-of-fold vector blends silently and wrongly.

In [ ]:
import gc
import hashlib
import time
from pathlib import Path

import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

## Stage 2. The representation, and the leak argument

The 12 raw columns. Categoricals stay categorical, NaNs stay NaN, and every learner handles
both natively. **Nothing here touches the target**, which is the whole leak argument and is
asserted below rather than described: the feature frames must be bit-identical under a
permutation of `y`.

That is a stronger statement than the checks in `13_target_encoding.ipynb` could make. Those
had to bound how much of a row's own target reached its own encoding, because an encoder was
running. Here there is nothing to bound.

In [ ]:
def frame(df, for_cb=False):
    X = df[COLS].copy()
    for c in CAT:
        if for_cb:
            # CatBoost takes categoricals as strings and will not accept NaN in them.
            X[c] = X[c].astype(object).fillna("__missing__").astype(str)
        else:
            X[c] = X[c].astype("category")
    return X


X = frame(train)
X_test = frame(test)
X_cb = frame(train, for_cb=True)
X_test_cb = frame(test, for_cb=True)
CAT_IDX = [X_cb.columns.get_loc(c) for c in CAT]

# Permute the target and rebuild. If the feature build touches y at all, the frames
# differ. DataFrame.equals treats NaN in the same position as equal, which plain `==`
# does not: on pandas 3.0 `astype(str)` preserves NA, so an `==` comparison silently
# reports every missing cell as unequal. That is the same version trap that eats target
# encodings, met here in a check rather than in a feature.
rng = np.random.default_rng(0)
train_perm = train.copy()
train_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
pure = (frame(train_perm).equals(X) and frame(train_perm, for_cb=True).equals(X_cb))
print(f"feature frame is a pure function of the features, not of y: {pure}")
print(f"target columns present in the feature set: "
      f"{[c for c in X.columns if c in ('id', TARGET)]}")
print(f"{len(COLS)} raw columns, {len(CAT)} categorical, no encoded columns")
print()
print("missing per column:")
for c in COLS:
    print(f"  {c:<26s}{train[c].isna().mean():>7.2%}")
CLEAN = pure and LEAK_OK
print()
print(f"leak checks: {'PASS' if CLEAN else 'FAILED'}")

## Stage 3. The three learners

Each arm's settings are its baseline row's settings. `lgb_raw` carries the determinism flags
that this repo records as mandatory, because LightGBM is not reproducible without them and
this repo found that the hard way.

In [ ]:
def make_lgb(n_est):
    # Row 9 and row 17's configuration.
    return lgb.LGBMClassifier(
        objective="binary", metric="auc", learning_rate=LR, n_estimators=n_est,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=SEED, n_jobs=THREADS, verbose=-1,
        deterministic=True, force_row_wise=True,
    )


def make_xgb(n_est):
    # Row 38's configuration.
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=THREADS, verbosity=0,
    )


def make_cat(n_est):
    # Row 26's configuration: library defaults apart from the budget.
    return cb.CatBoostClassifier(
        iterations=n_est, learning_rate=LR, random_seed=SEED,
        thread_count=THREADS, allow_writing_files=False, verbose=0,
    )


def run_fold(arm, fold, n_est, want_test=False):
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    t0 = time.time()
    if arm == "cat_raw":
        m = make_cat(n_est)
        m.fit(X_cb.iloc[tr], y[tr], cat_features=CAT_IDX)
        p = m.predict_proba(X_cb.iloc[va])[:, 1]
        p_te = m.predict_proba(X_test_cb)[:, 1] if want_test else None
    else:
        m = (make_lgb if arm == "lgb_raw" else make_xgb)(n_est)
        m.fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        p_te = m.predict_proba(X_test)[:, 1] if want_test else None
    out = {"va": va, "p": p, "p_te": p_te, "auc": float(roc_auc_score(y[va], p)),
           "secs": time.time() - t0}
    del m
    gc.collect()
    return out


def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


def note(msg):
    print(f"{time.strftime('%H:%M:%S')}  {msg}", flush=True)


bench = {}
for arm in ARMS:
    a = run_fold(arm, PROBE_FOLD, BENCH_EST)
    b = run_fold(arm, PROBE_FOLD, BENCH_EST)
    drift = abs(a["auc"] - b["auc"])
    bench[arm] = a["secs"]
    note(f"{arm}: bench {a['auc']:.6f} in {hhmm(a['secs'])}, "
         f"repeat drift {drift:.3e} {'OK' if drift == 0.0 else 'DIVERGED'}")

proj = sum(s / BENCH_EST * N_EST * 5 for s in bench.values())
print()
print(f"projected full run: {hhmm(proj)} for {len(ARMS)} arms x 5 folds")
if not SMOKE:
    assert proj < MAX_HOURS * 3600, (
        f"projected {hhmm(proj)} exceeds MAX_HOURS={MAX_HOURS}")
print("within budget" if proj < MAX_HOURS * 3600 else "SMOKE: guard not enforced")

## Stage 4. The full run

In [ ]:
results = {}
t_all = time.time()
for arm in ARMS:
    oof = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    scores = []
    t0 = time.time()
    for f in range(5):
        r = run_fold(arm, f, N_EST, want_test=True)
        oof[r["va"]] = r["p"]
        test_pred += r["p_te"] / 5
        scores.append(r["auc"])
        done = time.time() - t0
        note(f"  {arm} fold {f}: {scores[-1]:.6f}  ({hhmm(r['secs'])}), "
             f"elapsed {hhmm(done)}, about {hhmm(done / (f + 1) * (4 - f))} left")
    cv, sd = float(np.mean(scores)), float(np.std(scores))
    results[arm] = {"cv": cv, "sd": sd, "oof": oof, "test": test_pred,
                    "scores": np.array(scores)}
    print(f"{arm}: CV {cv:.6f} +/- {sd:.6f}   [{hhmm(time.time() - t0)}]")
    print()

print(f"all arms done in {hhmm(time.time() - t_all)}")

## Stage 5. The paired comparisons, and the row 9 reproduction

Each arm against the encoded row it changes one variable from. The bar is the spread of the
per-fold differences and the number of folds won, not the fold spread, which is common to
both and cancels.

`lgb_raw` additionally has a number it is required to hit: row 9 ran this exact
configuration on these exact folds. A reproduction failure here means something in the
environment moved and every arm in this notebook is suspect.

In [ ]:
for arm in ARMS:
    key, base_cv, label = BASELINES[arm]
    base = np.load(locate(f"{key}_oof.npy"))[ROW_IDX]
    bf = np.array([roc_auc_score(y[folds == f], base[folds == f]) for f in range(5)])
    d = results[arm]["scores"] - bf
    print(f"{arm} (no encoder) vs {label} (encoder on)")
    print(f"  baseline re-scored here {bf.mean():.6f}, "
          f"{bf.mean() - base_cv:+.2e} from its ledger number")
    print(f"  per-fold: {np.round(d, 6).tolist()}")
    print(f"  mean {d.mean():+.6f}, paired sd {d.std(ddof=1):.6f}, "
          f"wins {(d > 0).sum()}/5 folds")
    print()

rep = results["lgb_raw"]["cv"] - ROW9_CV
print(f"lgb_raw vs row 9 (the identical configuration): {rep:+.2e}")
if SMOKE:
    print("SMOKE: subsampled, so this is NOT the reproduction check. Not run.")
else:
    print("row 9 REPRODUCED" if abs(rep) < 1e-5 else
          "row 9 NOT reproduced - investigate before using any arm here")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
for arm, r in results.items():
    np.save(OUT / f"{pre}{arm}_oof.npy", r["oof"])
    np.save(OUT / f"{pre}{arm}_test.npy", r["test"])
    print(f"wrote {pre}{arm}_oof.npy, {pre}{arm}_test.npy")

print()
print("ledger lines:")
for arm, r in results.items():
    print(f"  name    {arm}")
    print(f"  cv_mean {r['cv']:.6f}")
    print(f"  cv_std  {r['sd']:.6f}")
print()
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}")
print()
print("No submission csv. Membership is 37_stack_views.ipynb, a separate ledger row.")